In [1]:
import pandas as pd
import re
from datetime import datetime
import numpy as np
from plotnine import ggplot, geom_bar, aes, geom_col, labs, theme, element_blank, element_text, facet_wrap, geom_line
import os

In [2]:
# Got GISAID data from Louise

In [3]:
LATESTGIS = "./metadata/sequence_metadata/GISAID_metadata.tsv"

In [4]:
data_gisaid = pd.read_csv(LATESTGIS, sep = "\t")

FileNotFoundError: [Errno 2] No such file or directory: './metadata/sequence_metadata/GISAID_metadata.tsv'

In [ ]:
data_gisaid.head()

In [ ]:
print(len(data_gisaid))

In [ ]:
# Step 1: Clean and organize metadata

In [ ]:
data_gisaid["region"].value_counts()

In [ ]:
# Filter for region = Europe, South America, China, Japan Korea, Southeast Asia

#data_gisaid_filtered_1 = data_gisaid[(data_gisaid["region"] == "Europe") | (data_gisaid["region"] == "South America") | (data_gisaid["region"] == "China") | (data_gisaid["region"] == "Japan Korea")]
data_gisaid_filtered_1 = data_gisaid[(data_gisaid["region"] == "Europe") |(data_gisaid["region"] == "South America") | (data_gisaid["region"] == "China") | (data_gisaid["region"] == "Japan Korea") | (data_gisaid["region"] == "Southeast Asia")]

data_gisaid_filtered_2 = data_gisaid_filtered_1[data_gisaid_filtered_1["host"] == "Avian"]

In [ ]:
# Check work

print(data_gisaid_filtered_2["region"].value_counts())
print(data_gisaid_filtered_2["host"].value_counts())
print(len(data_gisaid_filtered_2))

In [ ]:
# Filter for date = 2021, 2022, 2023, 2024

date_split = data_gisaid_filtered_2["date"].str.split(pat = "-", expand = True)
data_gisaid_filtered_3 = data_gisaid_filtered_2.copy()
data_gisaid_filtered_3["year"] = date_split[0]
data_gisaid_filtered_3["month"] = date_split[1]

years_of_outbreak = ("2021", "2022", "2023", "2024")

data_gisaid_filtered_current = data_gisaid_filtered_3[data_gisaid_filtered_3["year"].isin(years_of_outbreak)]

In [ ]:
# Check work

print(len(data_gisaid_filtered_current))

print(data_gisaid_filtered_current["year"].value_counts())

In [ ]:
data_gisaid_filtered_current.head()

In [ ]:
# First, remove samples with just collection year (no month or day)

data_gisaid_filtered_current = data_gisaid_filtered_current[~data_gisaid_filtered_current["date"].str.match(r"^\d{4}-XX-XX$")]
print(len(data_gisaid_filtered_current))
print(data_gisaid_filtered_current["region"].value_counts())

In [ ]:
# Standardize strain name

gisaid_filtered_current_strain = data_gisaid_filtered_current.copy()

gisaid_filtered_current_strain["strain"] = gisaid_filtered_current_strain["strain"].str.lower().replace("-","").replace(" ","").replace("_","")

In [ ]:
# Pull host and location from strain name

strain_split = gisaid_filtered_current_strain["strain"].str.split(pat = "/", expand = True)

gisaid_filtered_current_strain["strain_location"] = strain_split[2]

gisaid_filtered_current_strain["host_fromstrain"] = strain_split[1]

In [ ]:
# Determine which hosts are present

gisaid_global_avian_hosts = pd.DataFrame(set(gisaid_filtered_current_strain["host_fromstrain"]))

print(len(gisaid_global_avian_hosts)) #260 species

gisaid_global_avian_hosts.to_csv("./metadata/helpful_lists/gisaid_global_avian_hosts.csv", sep = ',')

In [ ]:
# Plot the hosts present in the GISAID data

gisaid_avian_hosts = pd.DataFrame(gisaid_filtered_current_strain["host_fromstrain"])

# plot this using ggplot
(
    ggplot(gisaid_avian_hosts, aes(x = "host_fromstrain"))
    + geom_bar(stat = "count")
    + labs(x = "Host Genus or Species", y = "Number of Sequences")
    + theme(axis_text_x = element_text(rotation = 90))
    + theme(figure_size=(15,5))
)

In [ ]:
# # SKIPPED THIS FOR NOW BUT CAN RETURN TO IT
# # Add columns on host (bird) taxonomic order and domestic status

# birdies = pd.read_csv("./metadata/helpful_lists/globaldataset_avian_hosts.csv", sep = ",", encoding='latin-1')

# gisaid_withbirds = pd.merge(gisaid_filtered_current_strain, birdies, how = "left", on="strain_host")

# print(len(gisaid_withbirds))

In [ ]:
# Add a NAflyway column that basically shows which NA flyway the country is closest to

# global_flyway_dict_1 = {"United Kingdom":"EastAtlanticFlyway",
#                       "Netherlands":,
#                       "Czech Republic":,
#                       "Spain":,
#                       "Japan":,
#                       "Poland":,
#                       "Italy":,
#                       "Belgium":,
#                       "South Korea":,
#                       "Austria":,
#                       "China":,
#                       "France":,
#                       "Germany":,
#                       "Chile":,
#                       "Argentina":,
#                       "Peru":,
#                       "Finland":,
#                       "Romania":,
#                       "Denmark":,
#                       "Croatia":,
#                       "Slovakia":,
#                       "Slovenia":,
#                       "Norway":,
#                       "Bolivia":,
#                       "Latvia":,
#                       "Uruguay":,
#                       "Moldova":,
#                       "Greece":,
#                       "Bulgaria":,
#                       "Switzerland":,
#                       "Luxembourg":,
#                       "Ecuador":,
#                       "Kosovo":,
#                       "Colombia":,
#                       "Venezuela":,
#                       "Sweden":,
#                       "Ireland":,
#                       "Albania":,
#                       "Iceland":,
#                       "Lithuania":,
#                       "Hong Kong":,
#                       "Brazil":, 
#                       "Ukraine":,
#                       "Serbia":,
#                       "Montenegro":,
#                       "Estonia":
#                      }

print(gisaid_filtered_current_strain["region"].value_counts())

global_region_dict_2 = {"Europe":"Europe",
                        "Japan Korea":"Asia",
                        "China":"Asia",
                        "South America":"SouthAmerica",
                       "Southeast Asia":"Asia"}

In [ ]:
# Add in region as a flyway column

gisaid_globalmvmt = gisaid_filtered_current_strain.copy()

gisaid_globalmvmt["NAflyway"] = gisaid_filtered_current_strain["region"].map(global_region_dict_2)

In [ ]:
# Make all the ? in "domestic_status" into "unknown" (otherwise it will mess up Nextstrain)

gisaid_globalmvmt["domestic_status"] = gisaid_globalmvmt["domestic_status"].str.replace("?","unknown")

In [ ]:
# List columns

print(gisaid_globalmvmt.columns)

In [ ]:
# Drop any unnecessary columns and rename any others

drop_columns = ["virus", "originating_lab", "submitting_lab", "authors", "PMID", "division"]

gisaid_globalmvmt_updt = gisaid_globalmvmt.drop(labels=drop_columns, axis = 1)

In [ ]:
# Change the "strain" column to "strain_title"

gisaid_globalmvmt_updt = gisaid_globalmvmt_updt.rename({"strain":"strain_title", "strain_location":"division"}, axis = 1)

In [ ]:
print(gisaid_globalmvmt_updt.columns)

In [ ]:
# Check for duplicated strain names

gisaid_globalmvmt_updt["strain_title"].is_unique

In [ ]:
# Function 2: Detemining duplicate strains
def to_dedupe(df):
    strain_count = df["strain_title"].value_counts()
    strain_count = strain_count.rename_axis("strain_title").reset_index()
    duplicate_strains = []
    for strain, count in strain_count.iterrows():
        if count['count'] > 1:
            duplicate_strains.append(count['strain_title'])
    duplicates = pd.DataFrame(duplicate_strains)
    return(duplicates)

In [ ]:
# Check which strains are duplicated

duplicates = gisaid_globalmvmt_updt.pipe(to_dedupe)

print(duplicates)

duplicated_GISglobal = gisaid_globalmvmt_updt[gisaid_globalmvmt_updt["strain_title"].isin(duplicates[0])]

duplicated_GISglobal.to_csv("./metadata/sequence_metadata/GISAID_global_duplicates.tsv", sep = '\t')

In [ ]:
# Remove duplicated strains

GISglobal_remove = [37458, #a/duck/cambodia/f1ppchba241d6/2021
                    29384, #a/duck/cambodia/f6t241d4/2021
                    22402, #a/duck/cambodia/f4k241d3c/2021
                    23191 #a/duck/cambodia/f4k241d4/2021
                   ]

gisaid_globalmvmt_updt_deduped = gisaid_globalmvmt_updt.drop(labels = GISglobal_remove, axis = 0)

In [ ]:
# How many sequences from each region?

print(gisaid_globalmvmt_updt_deduped["NAflyway"].value_counts())

print(gisaid_globalmvmt_updt_deduped["domestic_status"].value_counts())

In [ ]:
# Add in a new "strain" column to match nextstrain

gisaid_globalmvmt_updt_deduped["strain"] = gisaid_globalmvmt_updt_deduped[["strain_title", "date", "domestic_status", "NAflyway"]].astype(str).apply('|'.join, axis=1)

In [ ]:
# Save as an updated TSV

gisaid_globalmvmt_updt_deduped.to_csv("./metadata/sequence_metadata/GISAID_global_09182024_updated.tsv", sep = '\t')

In [ ]:
# In order to split this into segments and merge with the FASTAs to create new headers...
# We have to determine which strains have which segments

In [ ]:
#Function: Read in the FASTA as a dataframe
def fasta_reader(path_to_fasta, output_name):
    fasta_data = []
    FASTANAME = path_to_fasta
        
    with open(FASTANAME) as f:
        header = ""
        sequence = ""
        for line in f:
            if line.startswith(">"):
                if header != "":
                    fasta_data.append({"header": header, "sequence": sequence})
                header = line.strip() 
                sequence = ""
            else:
                sequence += line.strip()
        fasta_data.append({"header": header, "sequence": sequence}) #last line 
        
    globals()[output_name] = pd.DataFrame(fasta_data)

    return

In [ ]:
fasta_reader("./raw_data/GISAID_sequences_pb2.fasta", "gisaid_pb2")
fasta_reader("./raw_data/GISAID_sequences_pb1.fasta", "gisaid_pb1")
fasta_reader("./raw_data/GISAID_sequences_pa.fasta", "gisaid_pa")
fasta_reader("./raw_data/GISAID_sequences_ha.fasta", "gisaid_ha")
fasta_reader("./raw_data/GISAID_sequences_np.fasta", "gisaid_np")
fasta_reader("./raw_data/GISAID_sequences_na.fasta", "gisaid_na")
fasta_reader("./raw_data/GISAID_sequences_mp.fasta", "gisaid_m")
fasta_reader("./raw_data/GISAID_sequences_ns.fasta", "gisaid_ns")

In [ ]:
#Function: alter the FASTA headers
def fasta_headers(fasta_df):
    
    new_fasta_df = fasta_df.copy()
    
    new_fasta_df["strain_title"] = fasta_df["header"].str.lower()
    new_fasta_df["strain_title"] = new_fasta_df["strain_title"].str.replace(pat='-', repl='', regex=False)
    new_fasta_df["strain_title"] = new_fasta_df["strain_title"].str.replace(pat=' ', repl='', regex=False)
    new_fasta_df["strain_title"] = new_fasta_df["strain_title"].str.replace(pat='_', repl='', regex=False)
    new_fasta_df["strain_title"] = new_fasta_df["strain_title"].str.replace(pat='>', repl='', regex=False)

    return(new_fasta_df)

In [ ]:
# Alter FASTA headers

renamed_gisaid_pb2 = fasta_headers(gisaid_pb2)
renamed_gisaid_pb1 = fasta_headers(gisaid_pb1)
renamed_gisaid_pa = fasta_headers(gisaid_pa)
renamed_gisaid_ha = fasta_headers(gisaid_ha)
renamed_gisaid_np = fasta_headers(gisaid_np)
renamed_gisaid_na = fasta_headers(gisaid_na)
renamed_gisaid_m = fasta_headers(gisaid_m)
renamed_gisaid_ns = fasta_headers(gisaid_ns)

In [ ]:
# Function: Filter the FASTA file to just the set of strains in the metadata
def filter_fasta(renamed_fasta_df):

    metadata_strains = set(gisaid_globalmvmt_updt["strain_title"])

    renamed_filtered_fastadf = renamed_fasta_df[renamed_fasta_df["strain_title"].isin(metadata_strains)]

    return(renamed_filtered_fastadf)

In [ ]:
renamed_filtered_gisaid_pb2 = filter_fasta(renamed_gisaid_pb2)
renamed_filtered_gisaid_pb1 = filter_fasta(renamed_gisaid_pb1)
renamed_filtered_gisaid_pa = filter_fasta(renamed_gisaid_pa)
renamed_filtered_gisaid_ha = filter_fasta(renamed_gisaid_ha)
renamed_filtered_gisaid_np = filter_fasta(renamed_gisaid_np)
renamed_filtered_gisaid_na = filter_fasta(renamed_gisaid_na)
renamed_filtered_gisaid_m = filter_fasta(renamed_gisaid_m)
renamed_filtered_gisaid_ns = filter_fasta(renamed_gisaid_ns)

In [ ]:
# Function: Make a dictionary of lists
def list_strainspersegment(renamed_filtered_df, key_name):

    segment_name = []

    for i in renamed_filtered_df["strain_title"]:
        segment_name.append(i)

    dict_strains[key_name] = segment_name

    return

In [ ]:
dict_strains = {}

list_strainspersegment(renamed_filtered_gisaid_pb2, "1")
list_strainspersegment(renamed_filtered_gisaid_pb1, "2")
list_strainspersegment(renamed_filtered_gisaid_pa, "3")
list_strainspersegment(renamed_filtered_gisaid_ha, "4")
list_strainspersegment(renamed_filtered_gisaid_np, "5")
list_strainspersegment(renamed_filtered_gisaid_na, "6")
list_strainspersegment(renamed_filtered_gisaid_m, "7")
list_strainspersegment(renamed_filtered_gisaid_ns, "8")

In [ ]:
# Function: duplicate rows and assign segment value
def duplicate_rows_based_on_segment(df, dict_strains):
    # Initialize an empty list to store the new rows
    new_rows = []

    # Iterate through each row in the dataframe
    for idx, row in df.iterrows():
        # Check each segment in the dictionary
        for segment, strains in dict_strains.items():
            # Check if the strain is in the current segment's strains
            if row['strain_title'] in strains:
                new_row = row.copy()  # Copy the original row
                new_row['segment_no'] = segment  # Assign the segment
                new_rows.append(new_row)  # Append the new row to the list

    # Create a new dataframe from the list of new rows
    df_expanded = pd.DataFrame(new_rows)
    return df_expanded

In [ ]:
gisaid_global_expanded = duplicate_rows_based_on_segment(gisaid_globalmvmt_updt_deduped, dict_strains)

In [ ]:
print(len(gisaid_global_expanded))

In [ ]:
# Now, we have the following:
# A new, expanded metadata file with segment identity corresponding to the FASTAs
# Filtered FASTA files for each segment with an additional "strain" column that can be used to merge

In [ ]:
# Print the full, expanded GISAID metadata

# DATE = "09182024"

# gisaid_expanded.to_csv("./metadata/sequence_metadata/GISAIDdata_" + DATE + "_expanded_deduped.tsv", sep = '\t')

In [ ]:
# Divide by segment

segment_pb2 = gisaid_global_expanded[gisaid_global_expanded["segment_no"] == '1'] # Encodes PB2 
segment_pb1 = gisaid_global_expanded[gisaid_global_expanded["segment_no"] == '2'] # Encodes PB1 and sometimes PB1-F2
segment_pa = gisaid_global_expanded[gisaid_global_expanded["segment_no"] == '3'] # Encodes PA
segment_ha = gisaid_global_expanded[gisaid_global_expanded["segment_no"] == '4'] # Encodes HA
segment_np = gisaid_global_expanded[gisaid_global_expanded["segment_no"] == '5'] # Encodes NP
segment_na = gisaid_global_expanded[gisaid_global_expanded["segment_no"] == '6'] # Encodes NA
segment_m = gisaid_global_expanded[gisaid_global_expanded["segment_no"] == '7'] # Encodes M1 matrix
segment_ns = gisaid_global_expanded[gisaid_global_expanded["segment_no"] == '8'] # Encodes NS1, NEP, and NS2

print(len(segment_pb2))
print(len(segment_pb1))
print(len(segment_pa))
print(len(segment_ha))
print(len(segment_np))
print(len(segment_na))
print(len(segment_m))
print(len(segment_ns))

In [ ]:
# Print each segmented metadata as a TSV
segment_pb2.to_csv("./metadata/sequence_metadata/gisaid_global_pb2.tsv", sep = '\t')
segment_pb1.to_csv("./metadata/sequence_metadata/gisaid_global_pb1.tsv", sep = '\t')
segment_pa.to_csv("./metadata/sequence_metadata/gisaid_global_pa.tsv", sep = '\t')
segment_ha.to_csv("./metadata/sequence_metadata/gisaid_global_ha.tsv", sep = '\t')
segment_np.to_csv("./metadata/sequence_metadata/gisaid_global_np.tsv", sep = '\t')
segment_na.to_csv("./metadata/sequence_metadata/gisaid_global_na.tsv", sep = '\t')
segment_m.to_csv("./metadata/sequence_metadata/gisaid_global_m.tsv", sep = '\t')
segment_ns.to_csv("./metadata/sequence_metadata/gisaid_global_ns.tsv", sep = '\t')

In [ ]:
# Now merge FASTA and metadata to make new headers

In [ ]:
#Function: merge FASTA and metadata and assign new headers
def make_new_fasta_headers(deduped_fasta, meta_df, merged_fastameta):
    
    merged_meta_fasta = meta_df.merge(deduped_fasta, how = "inner", on = "strain_title")

    temp_fasta = merged_meta_fasta.copy()

    temp_fasta["header"] = merged_meta_fasta[["strain_title", "date", "domestic_status", "NAflyway"]].astype(str).apply('|'.join, axis=1)
    temp_fasta["header"] = ">" + temp_fasta["header"]

    temp_fasta["sequence"] = temp_fasta["sequence"].str.upper()

    globals()[merged_fastameta] = temp_fasta.copy()

    return

In [ ]:
# Merge FASTA and metadata

make_new_fasta_headers(renamed_filtered_gisaid_pb2, segment_pb2, "merged_pb2_fastameta")
make_new_fasta_headers(renamed_filtered_gisaid_pb1, segment_pb1, "merged_pb1_fastameta")
make_new_fasta_headers(renamed_filtered_gisaid_pa, segment_pa, "merged_pa_fastameta")
make_new_fasta_headers(renamed_filtered_gisaid_ha, segment_ha, "merged_ha_fastameta")
make_new_fasta_headers(renamed_filtered_gisaid_np, segment_np, "merged_np_fastameta")
make_new_fasta_headers(renamed_filtered_gisaid_na, segment_na, "merged_na_fastameta")
make_new_fasta_headers(renamed_filtered_gisaid_m, segment_m, "merged_m_fastameta")
make_new_fasta_headers(renamed_filtered_gisaid_ns, segment_ns, "merged_ns_fastameta")

In [ ]:
#Function: write a new FASTA with updated header (function written by Maria)  
def fasta_writer(path, filename, df):
            
    try:  
        os.mkdir(path)

    except OSError as error:
        pass

    with open(f"{path}{filename}", "w") as f:
        for index, row in df.iterrows():
            f.write(f"{row['header']}\n")
            f.write(f"{row['sequence']}\n")

In [ ]:
# Write final FASTAs

fasta_writer("./raw_data/", "GISAID_global_pb2.fasta", merged_pb2_fastameta)
fasta_writer("./raw_data/", "GISAID_global_pb1.fasta", merged_pb1_fastameta)
fasta_writer("./raw_data/", "GISAID_global_pa.fasta", merged_pa_fastameta)
fasta_writer("./raw_data/", "GISAID_global_ha.fasta", merged_ha_fastameta)
fasta_writer("./raw_data/", "GISAID_global_np.fasta", merged_np_fastameta)
fasta_writer("./raw_data/", "GISAID_global_na.fasta", merged_na_fastameta)
fasta_writer("./raw_data/", "GISAID_global_m.fasta", merged_m_fastameta)
fasta_writer("./raw_data/", "GISAID_global_ns.fasta", merged_ns_fastameta)

In [3]:
# Combine metadata with N. American avian metadata (for Nextstrain)

list_of_genes = ["pb1", "pb2", "pa", "ha", "na", "mp", "np", "ns"]

genes = list_of_genes

for gene in genes:

    #NAm_path = f"./metadata/sequence_metadata/NCBI+BVBRC+GIS_{gene}.tsv"
    #NAm_path = f"./metadata/sequence_metadata/NCBI-BVBRC-GIS_{gene}.tsv"
    # NAm_path = f"./Nextstrain/2025-06-24_NAm_aves-wild_single-intro_county-priority/metadata/NCBI-BVBRC-GIS_{gene}_detectionsgeodata.tsv"
    # NAm_path = f"./Nextstrain/2025-06-25_NAm_aves-wild_contextual-no-SAm_county-priority/metadata/NCBI-BVBRC-GIS_{gene}_detectionsgeodata.tsv"
    # NAm_path = f"./Nextstrain/2025-10-02_NAm_aves-wild_contextual-no-SAm_no-suliformes-clade_group-by-detections/metadata/{gene}_detectionsgeodata_yrmondiv.tsv"
    NAm_path = f"./Nextstrain/2025-12-22_NAm_aves-wild_contextual-no-SAm_county-priority/metadata/NCBI-BVBRC_{gene}_detectionsgeodata.tsv"

    ncbibvbrcgis_NAm = pd.read_csv(NAm_path, sep = '\t')

    ncbibvbrcgis_NAm = ncbibvbrcgis_NAm.rename({"ncbibvbrc_date": "date"}, axis = 1)

    global_path = f"./metadata/sequence_metadata/metadata_for_global_build/gisaid_global_{gene}.tsv"
    # global_path = f"./Nextstrain/2025-06-24_NAm_aves-wild_single-intro_county-priority/metadata/european_context_{gene}.tsv"
    gis_global = pd.read_csv(global_path, sep = '\t')

    gis_global = gis_global.rename({"NAflyway": "ncbibvbrc_country"}, axis = 1)
    
    ncbibvbrcgis_NAm_gis_global = pd.concat([ncbibvbrcgis_NAm, gis_global], join = "outer", ignore_index = True)
    
    # ncbibvbrcgis_NAm_gis_global = ncbibvbrcgis_NAm_gis_global.drop_duplicates(subset  = "strain", keep = "first", inplace = True, ignore_index = False)

    # ncbibvbrcgis_NAm_gis_global.to_csv(f"./metadata/sequence_metadata/NCBI+BVBRC+GIS_NAm_GISglobal_{gene}.tsv",
    #                                           sep = '\t')
    # ncbibvbrcgis_NAm_gis_global.to_csv(f"./Nextstrain/2025-06-24_NAm_aves-wild_single-intro_county-priority/metadata/NAm_detectionsgeodata_european_context_{gene}.tsv",
    #                                           sep = '\t')
    # ncbibvbrcgis_NAm_gis_global.to_csv(f"./Nextstrain/2025-06-25_NAm_aves-wild_contextual-no-SAm_county-priority/metadata/NAm_detectionsgeodata_context_{gene}.tsv",
    #                                           sep = '\t')
    # ncbibvbrcgis_NAm_gis_global.to_csv(f"./Nextstrain/2025-10-02_NAm_aves-wild_contextual-no-SAm_no-suliformes-clade_group-by-detections/metadata/NAm_detectionsgeodata_yrmondiv_context_{gene}.tsv",
    #                                           sep = '\t')
    ncbibvbrcgis_NAm_gis_global.to_csv(f"./Nextstrain/2025-12-22_NAm_aves-wild_contextual-no-SAm_county-priority/metadata/NCBI-BVBRC_detectionsgeodata_gis-global_{gene}.tsv",
                                              sep = '\t')

In [11]:
test = pd.read_csv("./metadata/sequence_metadata/metadata_for_global_build/NCBI+BVBRC+GIS_NAm_GISglobal_m.tsv", sep = '\t')

print(test["strain"].is_unique)

True


/var/folders/8z/j5vh6t990rj40lqqxtsr69z00000gp/T/ipykernel_95956/2918904714.py:1: DtypeWarning: Columns (2,3,5,6,9,10,11,12,15,20,21,22,24,25) have mixed types. Specify dtype option on import or set low_memory=False.


In [4]:
# Graph the timeline, so you know

In [5]:
# Remove ambiguous dates

gisaid_global_expanded_nonambigdates = gisaid_global_expanded[~gisaid_global_expanded["date"].str.match(r"^\d{4}-\d{2}-XX$")]

NameError: name 'gisaid_global_expanded' is not defined

In [ ]:
print(len(gisaid_global_expanded))
print(len(gisaid_global_expanded_nonambigdates))

In [ ]:
# Function 1: making Collection Date a datetime object
def format_date(datestr):
    i = str(datestr)
    date_object = datetime.strptime(i, '%Y-%m-%d')
    return (date_object)

In [ ]:
# Make the dates datetime objects

gisaid_global_expanded_nonambigdates["date"] = gisaid_global_expanded_nonambigdates["date"].apply(format_date)

print(type(gisaid_global_expanded_nonambigdates["date"].dtype))

In [ ]:
# Then, graph by date, facetted by segment, colored by NAflyway (region)

(
    ggplot(gisaid_global_expanded_nonambigdates, aes(x = "date", color = "NAflyway"))
    + geom_line(stat = "count")
    + facet_wrap("segment_no")
    + labs(x = "Date", y = "Number of Sequence")
    + theme(axis_text_x = element_text(rotation = 90), figure_size = (15, 10))
)